In [ ]:
from tke_benchmark.knowledge_prober import TemporalProber

# Specify paths!
MY_MODEL_PATH = "huggingface_cache/models/Llama-3.2-3B-Instruct"
MY_MAP_PATH = "dataset/news/relations_map_full.json"

# Instantiate Prober
prober = TemporalProber(model_path=MY_MODEL_PATH, map_path=MY_MAP_PATH)

INPUT_DATA = "dataset/news/news_full.txt"
OUTPUT_DATA = "dataset/llama/known_pure_facts.json"

# Call instance method to start inference
# prober.run_probing(data_file=INPUT_DATA, output_file=OUTPUT_DATA)
prober.run_probing_batched(data_file=INPUT_DATA, output_file=OUTPUT_DATA, batch_size=64)

/root/miniconda3/envs/edit02/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Initializing Prober...
Device: cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!

Parsing raw TXT data from dataset/news/news_full.txt...
Filtered to 461329 valid facts.
Starting batched inference with batch_size=64...


  9%|▉         | 644/7209 [06:09<1:02:25,  1.75it/s]

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

from tke_benchmark.dataset_builder import TKEBenchmarkBuilder

if __name__ == "__main__":

    # Specify the absolute or relative path to the pre-trained model.
    # We use this model to probe the knowledge base to ensure the dataset only contains facts the model already knows.
    MY_MODEL_PATH = "./huggingface_cache/models/gpt-j-6B"

    # Initialize the automated dataset building engine
    builder = TKEBenchmarkBuilder(
        model_path=MY_MODEL_PATH,
        news_dir="dataset/news",
        output_dir="dataset/gpt",
        batch_size=64  # Inference Batch Size. Reduce this if you encounter Out-Of-Memory errors on 3B+ models.
    )

    # Launch the end-to-end dataset generation pipeline
    builder.build_all(
        min_chain_length=2,         # Minimum number of consecutive events required for a valid entity chain
        max_cases_per_chain=32,     # Adjustable. Controls max soft cases sampled per chain. Higher values yield more total samples but may over-sample certain chains, reducing dataset uniformity.
        min_day_gap_insert=2,       # Minimum day interval between two events to allow a fictional counterfactual node insertion
        min_day_gap_modify=1        # Minimum day interval between three consecutive events required to modify the middle event
    )

/root/miniconda3/envs/edit02/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


🚀 [TKE Builder] Starting Temporal Knowledge Editing Benchmark Construction
   ➡️ Base Model:  ./huggingface_cache/models/gpt-j-6B
   ➡️ Source Dir:  dataset/news
   ➡️ Output Dir:  dataset/gpt

[1/4] Probing model's known facts...
Initializing Prober...
Device: cuda


Some weights of the model checkpoint at ./huggingface_cache/models/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transfor

Model loaded successfully!

Parsing raw TXT data from dataset/news/news_full.txt...
Filtered to 461329 valid facts.
Starting batched inference with batch_size=64...


100%|██████████| 7209/7209 [1:30:49<00:00,  1.32it/s]



Probing complete! Found 12137 perfectly known facts.
Retention Rate: 2.63%

[2/4] Constructing temporal chains (Min Chain Length: 2)...
Loading probed facts from dataset/gpt/known_pure_facts.json...
Grouped into 2083 unique (Subject, Relation) combinations.

🎯 Temporal Chains Building Complete!
Total Valid Chains Found: 918
Total Events utilized in chains: 10972 / 12137

Top 3 Longest Chains:
[438 events] Media Personnel (Iran) -> Make_statement
[332 events] Mahmoud Ahmadinejad -> Make_statement
[297 events] Aladdin Boroujerdi -> Make_statement

[3/4] Extracting counterfactual use cases from temporal chains (Insert / Modify)...
Processing 918 chains...

🚀 Relation-Constrained TKE Benchmark Complete!
Total Insert Test Cases Generated: 5090
  - Hard Cases (Object Shifted): 114
  - Soft Cases (Object Kept):    4976
Total Modify Test Cases Generated: 4845
  - Hard Cases (Object Shifted): 136
  - Soft Cases (Object Kept):    4709

[4/4] Assembling professional-grade TKE Benchmark datasets.

TypeError: generate_benchmark() got an unexpected keyword argument 'pre_insert_output_path'

In [1]:
from tke_benchmark.build_chains import build_temporal_chains

INPUT_PATH = "dataset/llama/known_pure_facts.json"  # Known facts database of the model
OUTPUT_PATH = "dataset/llama/temporal_chains.json"  # Fact chains
MIN_CHAIN_LENGTH = 3  # Minimum threshold to form a chain (at least 2 events)
build_temporal_chains(INPUT_PATH, OUTPUT_PATH, MIN_CHAIN_LENGTH)

Loading probed facts from dataset/llama/known_pure_facts.json...
Grouped into 2282 unique (Subject, Relation) combinations.

🎯 Temporal Chains Building Complete!
Total Valid Chains Found: 633
Total Events utilized in chains: 9688 / 11689

Top 3 Longest Chains:
[322 events] UN Security Council -> Impose_embargo,_boycott,_or_sanctions
[300 events] China -> Express_intent_to_engage_in_diplomatic_cooperation_(such_as_policy_support)
[294 events] China -> Criticize_or_denounce


In [1]:
from tke_benchmark.generate_tke_benchmark import generate_test_cases
from tke_benchmark.generate_tke_benchmark import generate_benchmark

INPUT_PATH = "dataset/llama/temporal_chains.json"
OUTPUT_PATH_INSERT = "dataset/llama/tke_test_cases_insert.json"
OUTPUT_PATH_MODIFY = "dataset/llama/tke_test_cases_modify.json"

MAX_CASES_PER_CHAIN = 32
MIN_DAY_GAP_INSERT = 2
MIN_DAY_GAP_MODIFY = 1

generate_test_cases(INPUT_PATH, OUTPUT_PATH_INSERT, OUTPUT_PATH_MODIFY, max_cases=MAX_CASES_PER_CHAIN, min_day_gap_insert=MIN_DAY_GAP_INSERT, min_day_gap_modify=MIN_DAY_GAP_MODIFY)

generate_benchmark(
        test_cases_path=OUTPUT_PATH_INSERT,
        map_path="dataset/news/relations_map_full.json",
        output_path="dataset/llama/tke_benchmark_insert.json",
        pre_insert_output_path="dataset/llama/tke_benchmark_pre_insert.json"
)

generate_benchmark(
        test_cases_path=OUTPUT_PATH_MODIFY,
        map_path="dataset/news/relations_map_full.json",
        output_path="dataset/llama/tke_benchmark_modify.json",
        pre_modify_output_path="dataset/llama/tke_benchmark_pre_modify.json"
)

Processing 633 chains...

🚀 Relation-Constrained TKE Benchmark Complete!
Total Insert Test Cases Generated: 4862
  - Hard Cases (Object Shifted): 321
  - Soft Cases (Object Kept):    4541
Total Modify Test Cases Generated: 5025
  - Hard Cases (Object Shifted): 362
  - Soft Cases (Object Kept):    4663
Transforming 4862 cases from dataset/llama/tke_test_cases_insert.json into professional TKE benchmark...
Done! Saved 4862 structured benchmark items to dataset/llama/tke_benchmark_insert.json
Done! Saved 4862 structured pre-insert benchmark items to dataset/llama/tke_benchmark_pre_insert.json
Transforming 5025 cases from dataset/llama/tke_test_cases_modify.json into professional TKE benchmark...
Done! Saved 5025 structured benchmark items to dataset/llama/tke_benchmark_modify.json
Done! Saved 5025 structured pre-modify benchmark items to dataset/llama/tke_benchmark_pre_modify.json


In [3]:
from tke_benchmark.build_chains import build_temporal_chains

INPUT_PATH = "dataset/gpt/known_pure_facts.json"  # Known facts database of the model
OUTPUT_PATH = "dataset/gpt/temporal_chains.json"  # Fact chains
MIN_CHAIN_LENGTH = 2  # Minimum threshold to form a chain (at least 2 events)
build_temporal_chains(INPUT_PATH, OUTPUT_PATH, MIN_CHAIN_LENGTH)

Loading probed facts from dataset/gpt/known_pure_facts.json...
Grouped into 2083 unique (Subject, Relation) combinations.

🎯 Temporal Chains Building Complete!
Total Valid Chains Found: 918
Total Events utilized in chains: 10972 / 12137

Top 3 Longest Chains:
[438 events] Media Personnel (Iran) -> Make_statement
[332 events] Mahmoud Ahmadinejad -> Make_statement
[297 events] Aladdin Boroujerdi -> Make_statement


In [4]:
from tke_benchmark.generate_tke_benchmark import generate_test_cases
from tke_benchmark.generate_tke_benchmark import generate_benchmark

INPUT_PATH = "dataset/gpt/temporal_chains.json"
OUTPUT_PATH_INSERT = "dataset/gpt/tke_test_cases_insert.json"
OUTPUT_PATH_MODIFY = "dataset/gpt/tke_test_cases_modify.json"

MAX_CASES_PER_CHAIN = 10
MIN_DAY_GAP_INSERT = 2
MIN_DAY_GAP_MODIFY = 1

generate_test_cases(INPUT_PATH, OUTPUT_PATH_INSERT, OUTPUT_PATH_MODIFY, max_cases=MAX_CASES_PER_CHAIN, min_day_gap_insert=MIN_DAY_GAP_INSERT, min_day_gap_modify=MIN_DAY_GAP_MODIFY)

generate_benchmark(
        test_cases_path=OUTPUT_PATH_INSERT,
        map_path="dataset/news/relations_map_full.json",
        output_path="dataset/gpt/tke_benchmark_insert.json",
        pre_insert_output_path="dataset/gpt/tke_benchmark_pre_insert.json"
)

generate_benchmark(
        test_cases_path=OUTPUT_PATH_MODIFY,
        map_path="dataset/news/relations_map_full.json",
        output_path="dataset/gpt/tke_benchmark_modify.json",
        pre_modify_output_path="dataset/gpt/tke_benchmark_pre_modify.json"
)

Processing 918 chains...

🚀 Relation-Constrained TKE Benchmark Complete!
Total Insert Test Cases Generated: 3316
  - Hard Cases (Object Shifted): 114
  - Soft Cases (Object Kept):    3202
Total Modify Test Cases Generated: 2832
  - Hard Cases (Object Shifted): 38
  - Soft Cases (Object Kept):    2794
Transforming 3316 cases from dataset/gpt/tke_test_cases_insert.json into professional TKE benchmark...
Done! Saved 3316 structured benchmark items to dataset/gpt/tke_benchmark_insert.json
Done! Saved 3316 structured pre-insert benchmark items to dataset/gpt/tke_benchmark_pre_insert.json
Transforming 2832 cases from dataset/gpt/tke_test_cases_modify.json into professional TKE benchmark...
Done! Saved 2832 structured benchmark items to dataset/gpt/tke_benchmark_modify.json
Done! Saved 2832 structured pre-modify benchmark items to dataset/gpt/tke_benchmark_pre_modify.json
